# Fine-tuning run: the agent's two answers

This notebook shows one LoRA fine-tuning run of a small open model and how it scored on the blind
held-out questions. The data is generated and the ward is simulated. Nothing here is validated on
real patients.

**It never trains and it never loads a model.** It reads only `training/agent/runs/<run-id>.json`,
which `python -m research.agent_tuning.runlog` writes from the run folder. Every cell uses the
Python standard library.

In [1]:
RUN = "run1"

import html, json, math, statistics
from pathlib import Path

root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "training" / "agent" / "runs").is_dir()), None)
path = root / "training" / "agent" / "runs" / f"{RUN}.json" if root else None
DOC = None
if path is None:
    print("No folder with training/agent/runs was found above this notebook. The cells below show nothing.")
elif not path.is_file():
    print(f"The run log training/agent/runs/{RUN}.json does not exist yet. Build it with "
          "`python -m research.agent_tuning.runlog` (see training/agent/README.md). The cells below show nothing.")
else:
    DOC = json.loads(path.read_text())
    print(f"Run log: training/agent/runs/{RUN}.json, {path.stat().st_size} bytes, format {DOC['format_version']}")
    print(f"Run {DOC['run_id']}. {DOC['note']}")

Run log: training/agent/runs/run1.json, 67937 bytes, format 1
Run run1. Generated data, simulated ward, not validated.


In [2]:
try:
    from IPython.display import HTML, display
except ImportError:
    HTML = display = None

TEXT = []  # the plain text of every table shown, so the numbers can be checked without a browser
CELL = "text-align:left;vertical-align:top;white-space:pre-wrap;word-break:break-word"


def table(title, headers, rows):
    """Shows a table. Every cell is escaped: inputs and outputs are untrusted text."""
    cells = [["" if c is None else str(c) for c in r] for r in rows]
    TEXT.append(title + "\n" + "\n".join(" | ".join(r) for r in [headers, *cells]))
    if display:
        head = "".join(f"<th style='{CELL}'>{html.escape(h)}</th>" for h in headers)
        body = "".join("<tr>" + "".join(f"<td style='{CELL}'>{html.escape(c)}</td>" for c in r) + "</tr>" for r in cells)
        display(HTML(f"<p><b>{html.escape(title)}</b></p><table><tr>{head}</tr>{body}</table>"))
    else:
        print(title)
        for r in [headers, *cells]:
            print("  " + " | ".join(r))


def frac(a, b):
    return f"{a}/{b}"


def cut(s, n=300):
    s = "" if s is None else str(s)
    return s if len(s) <= n else s[:n] + " ... (cut for display)"


def span(seconds):
    h, rest = divmod(round(seconds), 3600)
    return f"{h} h {rest // 60} min ({seconds:.0f} s)"


print("Tables are shown as HTML." if display else "IPython is not available: tables are printed as text.")

Tables are shown as HTML.


## Run facts

In [3]:
if DOC:
    d, a, s = DOC["data"], DOC["adapter"], DOC["settings"]
    base, fw, t = DOC["base_model"], DOC["framework"], DOC["times"]
    weights = next((f for f in base["files"] if f["name"] == "model.safetensors"), None)
    steps, tr = DOC["steps"], DOC["training"]
    rows = [("run id", DOC["run_id"]), ("data", DOC["note"]),
            ("base model", f"{base['name']}, revision {base['revision']}, licence {base['licence']}"),
            ("base weights (model.safetensors)", f"{weights['bytes']} bytes, sha256 {weights['sha256']}" if weights
             else base.get("sha256_note", "not listed")),
            ("LoRA", ", ".join(f"{k} {v}" for k, v in a["lora"].items())),
            ("trainable parameters", a["trainable_parameters"]),
            ("optimizer and gradients", f"{(s.get('optimizer') or {}).get('name', 'not recorded')}; {s.get('gradients') or 'not recorded'}"),
            ("adapter_model.safetensors", "not saved yet" if a["adapter_model.safetensors"]["sha256"] is None else
             f"{a['adapter_model.safetensors']['bytes']} bytes, sha256 {a['adapter_model.safetensors']['sha256']}"),
            ("adapter location", a["location"])]
    for name, rec in d["files"].items():
        rows.append((name, f"{rec['lines']} lines, sha256 {rec['sha256']}, "
                     f"{'matches the file' if rec['matches_file'] else 'DOES NOT match the file'}"))
    rows += [("data seed", d["seed"]),
             ("training", f"planned steps {tr['planned_steps']}, steps done {tr['steps_done']}, "
             + ("stopped by plan: " + tr["stopped_reason"] if tr["stopped_by_plan"] else
                "finished" if tr["complete"] else "not finished, not stopped by plan")),
             ("settings", f"epochs {s['epochs']}, learning rate {s['learning_rate']}, max length {s['max_length']}, "
              f"per group {s['per_group']}, seed {s['seed']}, batch size {s['batch_size']}, loss on {s['loss']}"),
             ("framework", f"{fw['backend']}; " + ", ".join(f"{k} {v}" for k, v in fw["versions"].items())
              + f" (versions from: {fw['source']})"),
             ("hardware", DOC["hardware"]),
             ("wall time", f"{span(DOC['wall_time']['seconds'])} (from: {DOC['wall_time']['source']})"),
             ("started / finished", f"{t['started_at']} / {t['finished_at']} (from: {t['source']})"),
             ("commit", DOC["commit"]),
             ("script hashes", ", ".join(f"{k} {v[:12]}" for k, v in DOC["scripts_sha256"].items()))]
    table("Run facts", ["item", "value"], rows)
    groups = sorted(set(d["trained_on"]) | set(d["planned"]))
    table("Examples trained on, per group", ["group", "trained on", "planned in run.json"],
          [(g, d["trained_on"].get(g, 0), d["planned"].get(g, 0)) for g in groups])

item,value
run id,run1
data,"Generated data, simulated ward, not validated."
base model,"Qwen/Qwen2.5-0.5B-Instruct, revision 7ae557604adf67be50417f59c2c2f167def9a775, licence Apache-2.0"
base weights (model.safetensors),"988097824 bytes, sha256 fdf756fa7fcbe7404d5c60e26bff1a0c8b8aa1f72ced49e7dd0210fe288fb7fe"
LoRA,"lora_alpha 16, lora_dropout 0.05, r 8, target_modules ['q_proj', 'v_proj']"
trainable parameters,540672
optimizer and gradients,"Adam, decoupled weight decay, written out; mindtorch.autograd.value_and_grad"
adapter_model.safetensors,"2175168 bytes, sha256 6f9ceb056616416e512df99542b6b85bc543bb9c6a4a18d6d48029dd48801670"
adapter location,~/Desktop/Projects/biosentry-data/agent-tuning/run1/adapter-step200
heldout.jsonl,"400 lines, sha256 d16d2686e51beb85700d20592e3e652426316e8bbe7b364936c7a2ca9cf1dcdf, matches the file"


group,trained on,planned in run.json
planner/evaluate_rules,30,30
planner/run_reward_round,5,30
planner/sweep_baseline,6,26
question/alerts_per_nurse/baseline,12,30
question/alerts_per_nurse/plain,9,30
question/alerts_per_nurse/threshold,9,30
question/ambiguous_threshold,10,30
question/approve_apply_write,12,30
question/baseline_vs_news2,26,30
question/change_product,11,30


## Loss

In [4]:
def nice(v):
    """A round tick step so that about 5 ticks span v."""
    base = 10 ** math.floor(math.log10(v / 4))
    return next(base * m for m in (1, 2, 5, 10) if v / (base * m) <= 5)


def curve(steps, window=20):
    W, H, L, R, T, B = 760, 340, 58, 16, 16, 46
    xs = max(r[0] for r in steps)
    xt, ymax = nice(xs), max(r[3] for r in steps)
    yt = nice(ymax)
    top, right = math.ceil(ymax / yt) * yt, math.ceil(xs / xt) * xt
    px = lambda x: L + (W - L - R) * x / right
    py = lambda y: H - B - (H - B - T) * y / top
    colour = {"q": "#2563eb", "p": "#d97706"}
    out = [f'<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 {W} {H}" width="{W}" role="img" '
           f'aria-label="Loss per training step" style="max-width:100%;height:auto">']
    axis = 'stroke="currentColor" stroke-opacity="0.55"'
    text = 'fill="currentColor" font-size="11" font-family="sans-serif"'
    out.append(f'<line x1="{L}" y1="{T}" x2="{L}" y2="{H - B}" {axis}/><line x1="{L}" y1="{H - B}" x2="{W - R}" y2="{H - B}" {axis}/>')
    for i in range(int(round(right / xt)) + 1):
        x = i * xt
        out.append(f'<line x1="{px(x):.1f}" y1="{H - B}" x2="{px(x):.1f}" y2="{H - B + 4}" {axis}/>'
                   f'<text x="{px(x):.1f}" y="{H - B + 17}" text-anchor="middle" {text}>{x:g}</text>')
    for i in range(int(round(top / yt)) + 1):
        y = i * yt
        out.append(f'<line x1="{L - 4}" y1="{py(y):.1f}" x2="{L}" y2="{py(y):.1f}" {axis}/>'
                   f'<text x="{L - 8}" y="{py(y) + 4:.1f}" text-anchor="end" {text}>{y:.2g}</text>')
    out.append(f'<text x="{(L + W - R) / 2}" y="{H - 8}" text-anchor="middle" {text}>training step</text>'
               f'<text x="14" y="{(T + H - B) / 2}" text-anchor="middle" transform="rotate(-90 14 {(T + H - B) / 2})" {text}>loss (answer tokens)</text>')
    if any(len(r) > 5 for r in steps):
        pts = " ".join(f"{px(r[0]):.1f},{py(r[5]):.1f}" for r in steps)
        out.append(f'<polyline fill="none" stroke="#8a8a8a" stroke-width="1.5" stroke-dasharray="4 3" points="{pts}"/>')
        out.append(f'<line x1="{W - R - 120}" y1="{T + 36}" x2="{W - R - 100}" y2="{T + 36}" stroke="#8a8a8a" stroke-width="1.5" stroke-dasharray="4 3"/>'
                   f'<text x="{W - R - 94}" y="{T + 40}" {text}>frozen model (control)</text>')
    for k, name in (("q", "question"), ("p", "planner")):
        rows = [r for r in steps if r[1] == k]
        out += [f'<circle cx="{px(r[0]):.1f}" cy="{py(r[3]):.1f}" r="1.8" fill="{colour[k]}" fill-opacity="0.25"/>' for r in rows]
        if rows:
            avg = [(r[0], statistics.fmean(x[3] for x in rows[max(0, i - window + 1):i + 1])) for i, r in enumerate(rows)]
            out.append(f'<polyline fill="none" stroke="{colour[k]}" stroke-width="2" points="'
                       + " ".join(f"{px(x):.1f},{py(y):.1f}" for x, y in avg) + '"/>')
        y = T + 8 + (14 if k == "p" else 0)
        out.append(f'<line x1="{W - R - 120}" y1="{y - 4}" x2="{W - R - 100}" y2="{y - 4}" stroke="{colour[k]}" stroke-width="2"/>'
                   f'<text x="{W - R - 94}" y="{y}" {text}>{name} steps</text>')
    out.append("</svg>")
    cap = (f"Loss on the answer tokens, one point per step ({len(steps)} steps). Faint points are the raw loss; "
           f"each line is the mean of the last {window} steps of that task."
           + (" The grey dashed line is the loss of the frozen model on the same examples." if any(len(r) > 5 for r in steps) else ""))
    return "".join(out), cap


if DOC and DOC["steps"]:
    svg, caption = curve(DOC["steps"])
    if display:
        display(HTML(f"<figure style='margin:0'>{svg}<figcaption>{html.escape(caption)}</figcaption></figure>"))
    else:
        print("[loss curve: an SVG, shown only where IPython is available]")
        print(caption)
elif DOC:
    print("The run log has no steps yet.")

In [5]:
if DOC and DOC["steps"]:
    rows = []
    for k, name in (("q", "question"), ("p", "planner")):
        loss = [r[3] for r in DOC["steps"] if r[1] == k]
        w = min(50, len(loss) // 2)
        first, last = loss[:w], loss[len(loss) - w:] if w else []
        rows.append((name, len(loss), f"{statistics.fmean(first):.3f} (n {len(first)})" if first else "-",
                     f"{statistics.fmean(last):.3f} (n {len(last)})" if last else "-"))
    table("Mean loss of the first and last window of steps of each task (window = min(50, steps // 2), no overlap)",
          ["task", "steps", "first window", "last window"], rows)
    ctl, w = DOC["training"]["control"], DOC["adapter"]["weights"]
    rows = []
    if ctl:
        rows.append(("trained loss below frozen loss, steps after step 1", frac(ctl["trained_below_control"], ctl["after_step_1"])))
        rows.append(("mean of trained minus frozen loss", ctl["mean_trained_minus_control"]))
    else:
        rows.append(("frozen-model control", "not in the run log"))
    if w["parsed"]:
        B, A = w["lora_B"], w["lora_A"]
        rows += [("lora_B tensors not all zero", frac(B["not_all_zero"], B["tensors"])), ("mean absolute lora_B", B["mean_abs"]),
                 ("mean absolute lora_A", A["mean_abs"])]
    else:
        rows.append(("adapter weights", "not read: " + w["reason"]))
    end = DOC["adapter"]["b_mean_abs_from_run_end"]
    rows.append(("mean absolute lora_B recorded when the run ended", "not recorded" if end is None else end))
    table("Did it learn? The frozen-model control and the adapter weights (lora_B starts at exactly zero)", ["measure", "value"], rows)

task,steps,first window,last window
question,159,0.142 (n 50),0.059 (n 50)
planner,41,0.261 (n 20),0.067 (n 20)


measure,value
"trained loss below frozen loss, steps after step 1",199/199
mean of trained minus frozen loss,-0.43258
lora_B tensors not all zero,48/48
mean absolute lora_B,0.002236773
mean absolute lora_A,0.016895723
mean absolute lora_B recorded when the run ended,not recorded


## Before and after on the blind held-out subsample

The subsample is the first N examples of each group in the held-out file, the same rule for every row.
Counts are shown as a/b. "After the guard" is what the agent would do: it acts on an answer only when
the guard accepts it. The keyword matcher and the perfect answers are computed on the same examples.

In [6]:
def system_rows(task):
    names = (("untuned", "untuned (step 0)"), ("tuned", "tuned (last step)"),
             ("matcher", "keyword matcher"), ("perfect", "perfect answers (ceiling)"))
    for key, name in names:
        r = DOC["evaluations"][task].get(key)
        if key == "tuned" and r:
            name = f"tuned (step {r['checkpoint_step']})"
        yield key, name, r


def incomplete(name, r, width):
    return [name, f"incomplete: {r['answers']} of {r['of']} answers"] + ["-"] * (width - 2)


if DOC:
    for task in ("question", "planner"):
        for key in ("untuned", "tuned"):
            r = DOC["evaluations"][task][key]
            what = "no answers file" if not r["checked"] else (
                "no adapter" if r["adapter_sha256"] is None else f"adapter sha256 {r['adapter_sha256']}")
            print(f"{key} {task} answers: {what}" + (" (checked against the answers file's meta file)" if r["checked"] else ""))
    ev = DOC["evaluations"]["question"]
    head = ["answers", "n", "valid JSON", "exact accuracy", "exact where the matcher answers", "exact where it is silent",
            "refusal accuracy", "outside the allowed shapes", "acted right", "acted wrong", "refused by the guard",
            "refusal questions acted on"]
    rows = []
    for key, name, r in system_rows("question"):
        if not r["complete"]:
            rows.append(incomplete(name, r, len(head)))
            continue
        m = r["scores"]
        rows.append([name, m["n"], frac(m["valid_json"], m["n"]), frac(m["exact"], m["exact_of"]),
                     frac(m["matcher_answers_exact"], m["matcher_answers"]), frac(m["matcher_silent_exact"], m["matcher_silent"]),
                     frac(m["refused_ok"], m["negatives"]), frac(m["outside_schema"], m["n"]),
                     frac(m["answerable_acted_right"], m["positives"]), frac(m["answerable_acted_wrong"], m["positives"]),
                     frac(m["answerable_guard_refused"], m["positives"]), frac(m["refusal_acted"], m["negatives"])])
    table("Questions: before and after (exact and acted columns count the answerable questions, refusal columns the refusals)", head, rows)

untuned question answers: no adapter (checked against the answers file's meta file)
tuned question answers: adapter sha256 6f9ceb056616416e512df99542b6b85bc543bb9c6a4a18d6d48029dd48801670 (checked against the answers file's meta file)
untuned planner answers: no adapter (checked against the answers file's meta file)
tuned planner answers: adapter sha256 6f9ceb056616416e512df99542b6b85bc543bb9c6a4a18d6d48029dd48801670 (checked against the answers file's meta file)


answers,n,valid JSON,exact accuracy,exact where the matcher answers,exact where it is silent,refusal accuracy,outside the allowed shapes,acted right,acted wrong,refused by the guard,refusal questions acted on
untuned (step 0),78,53/78,2/30,2/16,0/14,0/48,73/78,2/30,0/30,28/30,0/48
tuned (step 200),78,78/78,13/30,6/16,7/14,48/48,0/78,7/30,0/30,23/30,0/48
keyword matcher,78,78/78,14/30,14/16,0/14,46/48,0/78,13/30,2/30,15/30,2/48
perfect answers (ceiling),78,78/78,30/30,16/16,14/14,48/48,0/78,20/30,0/30,10/30,0/48


In [7]:
if DOC:
    head = ["answers", "n", "valid JSON", "exact accuracy (acted right)", "outside the allowed shapes",
            "acted on another open experiment", "fell back to the rule planner"]
    rows = []
    for key, name, r in system_rows("planner"):
        if key == "matcher":
            rows.append([name, "n/a: the matcher reads questions only"] + ["-"] * (len(head) - 2))
        elif not r["complete"]:
            rows.append(incomplete(name, r, len(head)))
        else:
            m = r["scores"]
            rows.append([name, m["n"], frac(m["valid_json"], m["n"]), frac(m["exact"], m["exact_of"]),
                         frac(m["outside_schema"], m["n"]), frac(m["acted_other_open"], m["n"]), frac(m["fell_back"], m["n"])])
    table("Planner: before and after", head, rows)

answers,n,valid JSON,exact accuracy (acted right),outside the allowed shapes,acted on another open experiment,fell back to the rule planner
untuned (step 0),9,5/9,0/9,9/9,0/9,9/9
tuned (step 200),9,9/9,9/9,0/9,0/9,0/9
keyword matcher,n/a: the matcher reads questions only,-,-,-,-,-
perfect answers (ceiling),9,9/9,9/9,0/9,0/9,0/9


## Examples

The first 3 examples of each group (groups in name order), at most 40 rows. Inputs and outputs are
untrusted text and are shown escaped. Long text is cut for display only.

In [8]:
if DOC:
    seen, rows = {}, []
    for ex in sorted(DOC["examples"], key=lambda e: e["group"]):
        seen[ex["group"]] = seen.get(ex["group"], 0) + 1
        if seen[ex["group"]] <= 3 and len(rows) < 40:
            rows.append([ex["group"], cut(ex["input"]), cut(ex["expected"]), cut(ex["untuned_output"], 200),
                         ex["untuned_guard"] or "no answer", cut(ex["tuned_output"], 200) if ex["tuned_output"] is not None else "not run yet",
                         ex["tuned_guard"] or "no answer"])
    table(f"Examples ({len(rows)} of {len(DOC['examples'])})",
          ["group", "input", "expected", "untuned output", "untuned guard", "tuned output", "tuned guard"], rows)

group,input,expected,untuned output,untuned guard,tuned output,tuned guard
planner/evaluate_rules,"{""rounds"": [{""id"": ""S-3766"", ""experiment"": ""reward_round"", ""ok"": false, ""chosen_generation"": null, ""validation_reward"": null}, {""id"": ""S-3768"", ""experiment"": ""reward_round_no_temperature"", ""ok"": true, ""chosen_generation"": 17, ""validation_reward"": 0.2634}], ""open"": [{""experiment"": ""check_policy_no_te ... (cut for display)","{""tool"":""evaluate_rules"",""args"":{""dataset"":""ward-v2"",""seeds"":[1,2,3,4,5,6,7],""window"":""early"",""rules"":[{""rule"":""news2_chart_trigger""},{""rule"":""news2_or_reward_policy"",""run"":""S-3768"",""no_temperature"":true}]},""reason"":""R4""}","{ ""tool"": ""evaluate_rules"", ""args"": { ""dataset"": ""ward-v2"", ""seeds"": [1, 2, 3, 4, 5, 6, 7], ""rules"": { ""variant"": ""news2_chart_trigger"", ""run"": ... (cut for display)",fell back,"{""tool"":""evaluate_rules"",""args"":{""dataset"":""ward-v2"",""seeds"":[1,2,3,4,5,6,7],""window"":""early"",""rules"":[{""rule"":""news2_chart_trigger""},{""rule"":""news2_or_reward_policy"",""run"":""S-3768"",""no_temperature"":t ... (cut for display)",acted: evaluate_rules
planner/evaluate_rules,"{""rounds"": [{""id"": ""S-2216"", ""experiment"": ""reward_round"", ""ok"": true, ""chosen_generation"": 6, ""validation_reward"": -0.9978}, {""id"": ""S-2218"", ""experiment"": ""reward_round_no_temperature"", ""ok"": true, ""chosen_generation"": 14, ""validation_reward"": -0.683}, {""id"": ""S-2219"", ""experiment"": ""sweep_baselin ... (cut for display)","{""tool"":""evaluate_rules"",""args"":{""dataset"":""ward-v2"",""seeds"":[1,2,3,4,5,6,7],""window"":""early"",""rules"":[{""rule"":""news2_chart_trigger""},{""rule"":""news2_or_reward_policy"",""run"":""S-2218"",""no_temperature"":true}]},""reason"":""R4""}","{ ""tool"": ""evaluate_rules"", ""args"": { ""dataset"": ""ward-v2"", ""seeds"": [1, 2, 3, 4, 5, 6, 7], ""window"": ""early"" } }",fell back,"{""tool"":""evaluate_rules"",""args"":{""dataset"":""ward-v2"",""seeds"":[1,2,3,4,5,6,7],""window"":""early"",""rules"":[{""rule"":""news2_chart_trigger""},{""rule"":""news2_or_reward_policy"",""run"":""S-2218"",""no_temperature"":t ... (cut for display)",acted: evaluate_rules
planner/evaluate_rules,"{""rounds"": [{""id"": ""S-0796"", ""experiment"": ""reward_round"", ""ok"": true, ""chosen_generation"": 13, ""validation_reward"": -0.7582}, {""id"": ""S-0798"", ""experiment"": ""sweep_baseline"", ""ok"": true, ""chosen_k"": null, ""chosen_p"": null}, {""id"": ""S-0799"", ""experiment"": ""reward_round_no_temperature"", ""ok"": true, "" ... (cut for display)","{""tool"":""evaluate_rules"",""args"":{""dataset"":""ward-v2"",""seeds"":[1,2,3,4,5,6,7],""window"":""early"",""rules"":[{""rule"":""news2_chart_trigger""},{""rule"":""news2_or_reward_policy"",""run"":""S-0796"",""no_temperature"":false}]},""reason"":""R2""}","{ ""tool"": ""check_policy"", ""args"": { ""dataset"": ""ward-v2"", ""seeds"": [1, 2, 3, 4, 5, 6, 7], ""window"": ""early"" } }",fell back,"{""tool"":""evaluate_rules"",""args"":{""dataset"":""ward-v2"",""seeds"":[1,2,3,4,5,6,7],""window"":""early"",""rules"":[{""rule"":""news2_chart_trigger""},{""rule"":""news2_or_reward_policy"",""run"":""S-0796"",""no_temperature"":f ... (cut for display)",acted: evaluate_rules
planner/run_reward_round,"{""rounds"": [{""id"": ""S-3380"", ""experiment"": ""reward_round"", ""ok"": true, ""chosen_generation"": null, ""validation_reward"": null}, {""id"": ""S-3382"", ""experiment"": ""sweep_baseline"", ""ok"": true, ""chosen_k"": 15, ""chosen_p"": 0.9226}], ""open"": [{""experiment"": ""reward_round_no_temperature"", ""tool"": ""run_reward_ ... (cut for display)","{""tool"":""run_reward_round"",""args"":{""train_dataset"":""ward-v2-train"",""validate_dataset"":""ward-v2"",""no_temperature"":true},""reason"":""R3""}","{ ""tool"": ""run_reward_round"", ""args"": { ""train_dataset"": ""ward-v2-train"", ""validate_dataset"": ""ward-v2"", ""no_

## What this does and does not show

- The loss curve: the trained loss is below the frozen control on 199 of the 199 steps after the first, on the same
  examples in the same order. So the adapter learned the training examples. It does not show that
  the model answers new questions. The run stopped by plan at step 200 of 476.
- The before and after counts: after the guard the tuned model gets more answerable questions
  right than the untuned model, and fewer than the keyword matcher. The tuned model alone is worse
  than the rules. The subsample is 78 questions and 9 planner examples, one run, no error bars. It
  is a first look, not a measurement.
- The guard and the keyword matcher: the guard is the boundary, not the weights. It acts on an
  answer only when the answer passes every check. It cannot check that the model chose the right
  experiment. Where the matcher is silent, a wrong choice among the three fixed experiments can
  pass, and the doctor then gets an answer about another experiment. In this run neither model had
  a wrong experiment run. No answer can change a setting. The guard also drops some right answers,
  and where the matcher answers, the matcher wins. The model can add something only where the
  matcher is silent.
- The data is generated and the ward is simulated because the project holds no patient data. The
  questions come from our own frames; the held-out frames were written blind. So nothing here says
  how real clinicians ask, or what happens on a real ward.
- Before any claim about real patients: a larger blind question set with error bars, a full
  training pass, questions from real clinicians, and a clinical study with ethics approval.